# Build Process: how this was built, and where AI helped

This notebook is the build log for the Parker Aerospace Installed-Base Reliability & Warranty Intelligence demo. It covers the workflow, the AI tool used at each layer, where AI was the biggest force multiplier, and where human judgment overrode it. All code ran in this workspace (`genie_zeroops_mfg_catalog.default`) and is committed in this repo.

**AI tool:** Databricks Assistant (Genie Code), used inside notebooks, the SQL editor, the pipeline editor, the Genie space, the dashboard editor and the Databricks App source.

## 1. Workflow

Every component went through the same loop:

1. **Intent.** I describe the goal and constraints in plain English, for example: *"Forecast which Parker components will have a HIGH/CRITICAL or warranty-risk event next quarter. Features only from before the cutoff, no derived health columns."*
2. **Draft.** Genie Code writes the notebook cells, SQL, pipeline file, Genie instructions, app route or slide generator.
3. **Run and verify.** The draft runs against live Unity Catalog tables. Output is checked with a second, independent query (for example, recomputing a dashboard counter in SQL) and by looking for evidence that it is wrong, not just that it ran.
4. **Commit.** Verified work is committed to this Git folder and pushed to GitHub (`thesqlpro/Fe-Bar`).

**Build order:** synthetic data → Lakeflow medallion pipelines → Unity Catalog governance → ML → metric views → Lakebase → Genie space → AI/BI dashboard → Databricks App (dashboard, Genie chat, Vector Search manual search) → value deck.

**Round 2 (after review feedback):** the reviewer's feedback was pasted into Genie Code. It audited the repo against each gap, proposed a plan, and then each fix was made and verified before committing: Parker reframe, UC access controls as code, leakage-free forecast, before/after KPIs, two-audience deck, realism checks, Genie and dashboard alignment, and this build log.

## 2. What AI did at each layer

| Layer | Asset | What Genie Code drafted | What I verified or decided |
|---|---|---|---|
| Synthetic data | `Aircraft_Data_Simulator` | 1,640 Parker components on 328 aircraft at 10 operators; 9,839 service records; 6 anomaly types with severity→warranty/delay correlations and type-specific detection latency | Realism profile cell. Reported the weakness openly: overdue service does not raise anomaly risk |
| Pipelines | `pipeline/`, `pipeline_parker/` | Bronze/silver/gold Lakeflow SDP code with expectations | Join keys, null handling; registry rename only after tracing all dependencies and lineage |
| Governance | `UC_Governance`, `governance/uc_access_controls.sql` | Comments, quality tags, owner properties, entitlement table, row filter, column mask, grants, lineage query | Applied live; confirmed rows visible per principal (10/10 and 1,640/1,640 for the owner) |
| ML | `Parker_Component_Failure_Forecast` | Point-in-time features, future-window label, 5-fold stratified CV, MLflow logging, back-test | Rejected the 0.999 AUC as target leakage; picked top-decile capture as the business metric |
| Metric views | `Gold_Layer_Metric_Views` | YAML metric definitions | `part_failure_metrics` repointed from the retired leaky model to the forecast; MEASURE() check returns 1,640 / 164 / 164 |
| Lakebase | synced tables | Synced-table specs | Row-filtered tables cannot be synced; kept governance and synced the unfiltered Parker tables |
| Genie space | Aircraft Maintenance Intelligence | Instructions, filters, example SQL, benchmarks | Both Parker benchmarks pass (18 actuators; 164 visit-list components) |
| Dashboard | Parker Installed Base page | Datasets and widgets | Counters checked: 1,640 and 164 |
| App | `app/main.py` | Flask routes, Genie polling and charts, Vector Search manual tab | Replaced a starter question whose value did not exist in the data |
| Deck | `Parker_Aerospace_Value_Deck` | python-pptx generator with every number queried live | Set and disclosed the 60% intervention-success and $250K/year run-cost assumptions |

## 3. Where AI was the force multiplier

1. **Audit → fix → verify in one session.** Round-2 feedback covered six gaps across docs, governance, ML and the deck. Genie Code mapped each gap to files, made the fix and re-ran the checks in the same session.
2. **Challenging its own headline metric.** Asked to stress-test the 0.999 AUC, it traced the label (`failure_risk_category`) to a threshold on an input feature (`health_index`). It then rebuilt the model as a next-quarter forecast with a time-based split: AUC 0.60, top-decile capture 22-24% (2.2-2.4x random vs 1.2x for a rule).
3. **Governance as code.** The entitlement table, row filter, column mask and grants were generated, applied and verified quickly, so access control lives in the repo, not in a slide.
4. **Serving-layer configuration.** Genie instructions, example SQL and benchmark questions were drafted and scored automatically. Dashboard widgets were built on metric views and their counters checked against SQL.

## 4. Where human judgment overrode the AI

- **Story:** chose Parker's real position (Tier-1 OEM carrying warranty and AOG cost) and the KPI owners by role.
- **Model honesty:** refused to present the 0.999 AUC; accepted a weaker but honest model.
- **Assumptions:** set the 60% intervention-success rate and $250K/year run cost, and put them on the slides.
- **Safety:** approved the table rename, deletions and pushes only after dependency checks; kept row filters even though they block Lakebase sync for two tables.
- **Transparency:** reported the simulator's known weakness instead of tuning it away before review.